In [1]:
# %% [markdown]
# # 4 高级循环神经网络
# ## 4.1 理论计算题：深度双向RNN参数总量
# 设定：L层，每层隐藏单元H，输入维度D，输出维度O
# 单层单向RNN参数：
# W_hx(H,D), W_hh(H,H), b_h(H) → HD + H² + H
# 双向每层=前向+后向，单层双向参数 = 2*(HD + H² + H)
# L层双向总RNN层参数：L * 2*(HD + H² + H)
# 输出层：输入是2H（拼接双向最后一层），输出O
# 输出层参数：W_o(O,2H) + b_o(O) = 2HO + O
# 总参数表达式：
# Total = 2L(HD + H² + H) + 2HO + O

# %% [markdown]
# ## 4.2 编程题：双向RNN编码器（PyTorch）
# %%
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False
        )
    
    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        seq_out, h_n = self.rnn(X)
        # seq_out: (seq_len, batch, 2*hidden_dim) 每个时间步拼接前向后向
        # h_n: (2*num_layers, batch, hidden_dim)
        # 取最后一层前向、后向拼接作为全局序列表示
        last_forward = h_n[-2, :, :]
        last_backward = h_n[-1, :, :]
        global_repr = torch.cat([last_forward, last_backward], dim=-1)  # (batch, 2H)
        return seq_out, global_repr

# 测试
if __name__ == "__main__":
    seq_len, batch, d_in = 5, 3, 4
    d_h = 6
    model = BiRNNEncoder(input_dim=d_in, hidden_dim=d_h)
    x = torch.randn(seq_len, batch, d_in)
    seq_emb, global_emb = model(x)
    print("逐时间步输出 shape:", seq_emb.shape)  # (5,3,12)
    print("全局序列表示 shape:", global_emb.shape) # (3,12)

逐时间步输出 shape: torch.Size([5, 3, 12])
全局序列表示 shape: torch.Size([3, 12])
